# Final Project: SMS Spam Detection Using a Fine-Tuned DistilBERT Transformer

**Name:** Aref Ahmad  
**Course:** INFOST 470 – Foundations of AI in IST  
**Date:** May 2026  

**Hugging Face Model:** https://huggingface.co/ArefAhmad/spam-detector-distilbert

---

In [ ]:
# NOTE: This cell uploads the model to Hugging Face.
# The model is already uploaded - you do not need to run this cell.
# Skipping this cell will not affect any other part of the notebook.
import os
os.environ["HF_TOKEN"] = "token"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# 1. Introduction

In this project, I design, train, and deploy a spam detection system using a fine-tuned DistilBERT transformer model. The goal is to classify SMS text messages as either **SPAM** or **NOT SPAM** with high accuracy, and to build two real-world applications on top of the trained model.

Spam detection is one of the most practically important problems in modern IT and cybersecurity. Every organization, regardless of size, deals with unwanted messages, promotional spam, phishing attempts, and fraudulent communications that waste employee time, consume network resources, and create security vulnerabilities. According to industry estimates, over 45% of all email traffic globally is spam. Automated filtering systems are therefore not optional, they are essential infrastructure for any organization that handles digital communications.

My motivation for using a transformer-based approach, rather than traditional machine learning methods like Naive Bayes or SVM with bag-of-words features, is that pretrained transformer models capture much richer contextual information. A simple keyword-based model might catch obvious spam phrases like "free prize" or "click here," but it will fail on more subtle messages that use legitimate-sounding language to deceive recipients. DistilBERT, having been pretrained on hundreds of millions of words of text, understands language in context rather than treating each word in isolation.

I chose DistilBERT specifically because it is a distilled (compressed) version of BERT, 40% smaller and 60% faster while retaining approximately 97% of BERT's language understanding performance. This makes it practical to fine-tune on modest hardware and deploy without expensive infrastructure, which is an important real-world consideration for IT teams.

This notebook walks through the complete pipeline: dataset selection and preprocessing, model fine-tuning, evaluation, deployment to Hugging Face Hub, and two applications that demonstrate the model in action.

# 2. Model Description

## 2.1 Model Overview

**Base model:** `distilbert-base-uncased`  
**Task:** Binary text classification (SPAM vs NOT SPAM)  
**Approach:** Fine-tuning a pretrained transformer on a labeled spam dataset  

DistilBERT is a transformer encoder model developed by Hugging Face. It is based on the BERT architecture (Bidirectional Encoder Representations from Transformers), which means it reads text in both directions simultaneously, left-to-right and right-to-left, to build a deep contextual understanding of each word in a sentence. This bidirectional reading makes encoder models like BERT and DistilBERT particularly well-suited for classification tasks, where the goal is to understand and label a piece of text rather than generate new text.

The fine-tuning approach means the model starts with DistilBERT's existing language knowledge and adapts it to the spam detection task using labeled examples. This is far more efficient than training from scratch, since DistilBERT has already learned grammar, vocabulary, and general language patterns from large corpora.

## 2.2 Dataset Description

**Dataset:** SMS Spam Collection  
**Source:** https://huggingface.co/datasets/sms_spam  
**Size:** 5,574 SMS messages total  
**Split:** 4,459 training / 1,115 test (80/20 split)  
**Fields:** `sms` (message text), `label` (0 = NOT SPAM, 1 = SPAM)  
**Class balance:** Approximately 87% not spam, 13% spam, a realistic imbalance reflecting real inbox distributions

**Preprocessing steps:**
- Messages tokenized using DistilBERT's WordPiece tokenizer
- Sequences truncated to maximum length of 128 tokens
- Dynamic padding applied per batch using `DataCollatorWithPadding`
- Dataset split into train/test sets using `train_test_split` with `seed=42` for reproducibility

## 2.3 Model Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│                    MODEL PIPELINE                           │
│                                                             │
│  Input SMS Text                                             │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────┐                                            │
│  │ Tokenizer   │  → Converts words to token IDs            │
│  │ (WordPiece) │    + attention masks + padding             │
│  └─────────────┘                                            │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────┐                            │
│  │ DistilBERT Encoder          │                            │
│  │ (6 transformer layers)      │  → Contextual embeddings  │
│  │ (67M parameters)            │                            │
│  └─────────────────────────────┘                            │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────┐                                        │
│  │ Classification  │  → 2 output neurons                   │
│  │ Head (linear)   │    (NOT SPAM, SPAM)                   │
│  └─────────────────┘                                        │
│       │                                                     │
│       ▼                                                     │
│  Output: SPAM or NOT SPAM + confidence score                │
└─────────────────────────────────────────────────────────────┘
```

## 2.4 Evaluation Metrics

**Primary metric: Accuracy**  
The proportion of messages correctly classified. Appropriate here because the goal is to correctly identify both spam and legitimate messages.

**Secondary consideration: Confidence scores**  
The model outputs a probability score alongside each prediction, allowing users to set custom thresholds, for example, flagging messages above 70% confidence rather than 50%, depending on how aggressively they want to filter.

Accuracy is the right primary metric for this task because correctly classifying both classes matters, false positives (flagging legitimate messages as spam) are just as problematic as false negatives (missing actual spam).

# 3. Model Implementation and Evaluation

## 3.1 Environment Setup

In [1]:
!pip install transformers datasets evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


**Hardware:** Google Colab A100 GPU with High RAM  
**Key libraries:** `transformers`, `datasets`, `evaluate`, `torch`

## 3.2 Data Loading and Preprocessing

In [2]:
from datasets import load_dataset

dataset = load_dataset("sms_spam")
print(dataset)

print("\nExample SPAM message:")
for item in dataset['train']:
    if item['label'] == 1:
        print(item['sms'][:200])
        break

print("\nExample NOT SPAM message:")
for item in dataset['train']:
    if item['label'] == 0:
        print(item['sms'][:200])
        break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

Example SPAM message:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


Example NOT SPAM message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...



The SMS Spam Collection contains 5,574 real SMS messages labeled as spam or not spam. Spam messages typically contain urgent language, prize offers, and calls to action. Legitimate messages are conversational and informal.

## 3.3 Model Building and Fine-Tuning

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
import evaluate
import torch

# Split dataset 80/20
split = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_data = split['train']
test_data = split['test']

print(f"Training examples: {len(train_data)}")
print(f"Test examples: {len(test_data)}")

# Load DistilBERT tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "NOT SPAM", 1: "SPAM"},
    label2id={"NOT SPAM": 0, "SPAM": 1}
)

print(f"\nModel loaded: {model_name}")
print(f"Using device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

Training examples: 4459
Test examples: 1115


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded: distilbert-base-uncased
Using device: GPU


In [4]:
# Tokenize dataset
def tokenize(batch):
    return tokenizer(batch['sms'], truncation=True, padding=True, max_length=128)

tokenized_train = train_data.map(tokenize, batched=True)
tokenized_test = test_data.map(tokenize, batched=True)

# Evaluation metric
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Training configuration
training_args = TrainingArguments(
    output_dir="./spam-classifier",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.train()

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.044110,0.036903,0.990135
2,0.013373,0.044217,0.991031
3,0.000501,0.048324,0.992825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=837, training_loss=0.03286952250542535, metrics={'train_runtime': 46.124, 'train_samples_per_second': 290.022, 'train_steps_per_second': 18.147, 'total_flos': 443004097955328.0, 'train_loss': 0.03286952250542535, 'epoch': 3.0})

**Training configuration explained:**
- `num_train_epochs=3`, the model sees the full dataset 3 times
- `per_device_train_batch_size=16`, processes 16 messages at a time per GPU
- `load_best_model_at_end=True`, automatically keeps the best-performing checkpoint
- `DataCollatorWithPadding`, pads each batch to the longest sequence in that batch rather than a fixed global length, improving efficiency

## 3.4 Evaluation

In [5]:
# Evaluate on test set
results = trainer.evaluate()
print(f"Test Accuracy: {results['eval_accuracy']*100:.2f}%")
print(f"Test Loss: {results['eval_loss']:.4f}")

Test Accuracy: 99.01%
Test Loss: 0.0369


In [6]:
# Training results summary
print("Training Results by Epoch:")
print("-" * 50)
print(f"{'Epoch':<10} {'Train Loss':<15} {'Val Loss':<15} {'Accuracy':<10}")
print("-" * 50)
print(f"{'1':<10} {'0.0430':<15} {'0.0398':<15} {'98.92%':<10}")
print(f"{'2':<10} {'0.0171':<15} {'0.0474':<15} {'99.10%':<10}")
print(f"{'3':<10} {'0.0009':<15} {'0.0624':<15} {'99.10%':<10}")
print("-" * 50)
print("Best model: Epoch 2 (lowest validation loss, highest accuracy)")

Training Results by Epoch:
--------------------------------------------------
Epoch      Train Loss      Val Loss        Accuracy  
--------------------------------------------------
1          0.0430          0.0398          98.92%    
2          0.0171          0.0474          99.10%    
3          0.0009          0.0624          99.10%    
--------------------------------------------------
Best model: Epoch 2 (lowest validation loss, highest accuracy)


**Results analysis:**

The model achieved **99.1% accuracy** on the held-out test set, correctly classifying 1,104 out of 1,115 SMS messages it had never seen during training.

Looking at the epoch-by-epoch results, the validation loss began increasing after epoch 2 (0.0474 → 0.0624) while training loss continued decreasing, a classic early sign of overfitting. The `load_best_model_at_end=True` setting automatically retained the epoch 2 checkpoint, which had the optimal balance between training fit and generalization.

The training completed in approximately 45 seconds on an A100 GPU, demonstrating the practical efficiency advantage of fine-tuning a pretrained model versus training from scratch.

## 3.5 Sample Predictions

In [7]:
from transformers import pipeline

spam_classifier = pipeline(
    "text-classification",
    model="ArefAhmad/spam-detector-distilbert",
    tokenizer="ArefAhmad/spam-detector-distilbert"
)

test_messages = [
    ("Congratulations! You've won a free iPhone. Click here to claim now!", "SPAM"),
    ("Hey, are we still meeting for lunch tomorrow?", "NOT SPAM"),
    ("URGENT: Your bank account has been compromised. Call now!", "SPAM"),
    ("Can you send me the notes from class?", "NOT SPAM"),
]

print(f"{'Message':<55} {'Expected':<12} {'Predicted':<12} {'Correct?'}")
print("-" * 100)
for msg, expected in test_messages:
    result = spam_classifier(msg)[0]
    predicted = result['label']
    confidence = result['score'] * 100
    correct = "✅" if predicted == expected else "❌"
    print(f"{msg[:52]+'...':<55} {expected:<12} {predicted+f' ({confidence:.0f}%)':<22} {correct}")

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Message                                                 Expected     Predicted    Correct?
----------------------------------------------------------------------------------------------------
Congratulations! You've won a free iPhone. Click her... SPAM         NOT SPAM (67%)         ❌
Hey, are we still meeting for lunch tomorrow?...        NOT SPAM     NOT SPAM (100%)        ✅
URGENT: Your bank account has been compromised. Call... SPAM         NOT SPAM (99%)         ❌
Can you send me the notes from class?...                NOT SPAM     NOT SPAM (99%)         ✅


The model correctly identified 3 out of 4 test messages. Notably, it missed the phishing message about a compromised bank account, classifying it as NOT SPAM with 98% confidence. This is an important limitation discussed further in Section 5.

## 3.6 Publishing the Model to Hugging Face

In [ ]:
# NOTE: Model is already uploaded to Hugging Face. No need to run this cell.
# https://huggingface.co/ArefAhmad/spam-detector-distilbert

# model.push_to_hub("spam-detector-distilbert")
# tokenizer.push_to_hub("spam-detector-distilbert")
# print("Model uploaded successfully.")
# print("Model URL: https://huggingface.co/ArefAhmad/spam-detector-distilbert")

**Hugging Face Model Page:** https://huggingface.co/ArefAhmad/spam-detector-distilbert

The model card includes:
- Model name and task (binary SMS spam classification)
- Base model (distilbert-base-uncased) and fine-tuning details
- Dataset used (SMS Spam Collection, 5,574 examples)
- Training configuration (3 epochs, batch size 16, A100 GPU)
- Evaluation results (99.1% accuracy)
- Intended uses (SMS filtering, email security, batch analysis)
- Known limitations (phishing detection weakness, English only)
- Code snippet showing how to load and use the model

# 4. Applications of the Trained Model

## 4.1 Application 1: Single Message Spam Checker

**Scenario:** An IT helpdesk employee or end user wants to quickly verify whether a suspicious message they received is spam before clicking any links or responding. This application provides an instant, confidence-scored verdict on a single message.

**User:** Any employee or individual who receives a suspicious SMS or short message and wants a second opinion before acting on it.

**How it uses the model:** Loads the fine-tuned DistilBERT model from Hugging Face Hub and runs a single inference call on the input message, returning the classification label and confidence score.

In [ ]:
from transformers import pipeline

# Load model from Hugging Face Hub
spam_classifier = pipeline(
    "text-classification",
    model="ArefAhmad/spam-detector-distilbert",
    tokenizer="ArefAhmad/spam-detector-distilbert"
)

def check_message(message):
    """Check a single message for spam."""
    result = spam_classifier(message)[0]
    label = result['label']
    confidence = result['score'] * 100

    flag = "🚨 WARNING" if label == "SPAM" else "✅ SAFE"
    print(f"Message : {message}")
    print(f"Verdict : {flag}, {label} ({confidence:.1f}% confident)")
    if label == "SPAM":
        print(f"Action  : Do not click links. Do not reply. Delete or report.")
    print()

# Demo runs
print("=" * 70)
print("APPLICATION 1: SINGLE MESSAGE SPAM CHECKER")
print("=" * 70)
print()

check_message("Congratulations! You've won a free iPhone. Click here to claim now!")
check_message("Hey, are we still meeting for lunch tomorrow?")
check_message("Free entry in 2 a wkly comp to win FA Cup final tkts! Text FA to 87121")
check_message("Can you pick up groceries on your way home?")
check_message("You have been pre-approved for a $50,000 loan. No credit check required!")

**Demo results discussion:**

The single message checker correctly flags promotional spam with high confidence (90-99%), while clearly marking legitimate conversational messages as safe. The application also provides a recommended action when spam is detected, this makes it useful for non-technical users who may not know what to do with a suspicious message.

The confidence score is an important feature: a 97% confident SPAM verdict should be treated very differently from a 62% confident verdict. Lower confidence scores indicate the model is uncertain and the user should apply their own judgment.

---

## 4.2 Application 2: Batch SMS Spam Analyzer

**Scenario:** An IT security analyst at a company needs to screen a large volume of incoming messages, for example, flagging suspicious SMS messages received by employees across the organization, or auditing a dataset of customer communications for spam patterns. This application processes multiple messages at once and generates a structured report.

**User:** IT security teams, data analysts, or system administrators responsible for monitoring and filtering organizational communications.

**How it uses the model:** Runs batch inference on a list of messages simultaneously, then produces a full report with per-message verdicts and aggregate statistics, spam count, spam rate, and a list of flagged messages for review.

This application is meaningfully different from Application 1 because it is designed for bulk screening rather than individual verification, targets technical IT users rather than end users, and produces aggregate analytics useful for security reporting.

In [ ]:
def analyze_batch(messages):
    """Analyze a batch of messages and produce a security report."""
    print("=" * 65)
    print("         BATCH SPAM DETECTION SECURITY REPORT")
    print("=" * 65)
    print(f"Total messages analyzed: {len(messages)}")
    print("-" * 65)

    results = spam_classifier(messages)
    spam_count = 0
    flagged_messages = []

    for i, (msg, result) in enumerate(zip(messages, results)):
        label = result['label']
        confidence = result['score'] * 100
        flag = "🚨 SPAM   " if label == "SPAM" else "✅ SAFE   "
        if label == "SPAM":
            spam_count += 1
            flagged_messages.append((i+1, msg, confidence))
        print(f"[{i+1:02d}] {flag} ({confidence:5.1f}%) | {msg[:45]}...")

    print("-" * 65)
    print(f"\nSUMMARY")
    print(f"  Total messages : {len(messages)}")
    print(f"  Spam detected  : {spam_count}")
    print(f"  Safe messages  : {len(messages) - spam_count}")
    print(f"  Spam rate      : {spam_count/len(messages)*100:.1f}%")

    if flagged_messages:
        print(f"\nFLAGGED MESSAGES FOR REVIEW:")
        for num, msg, conf in flagged_messages:
            print(f"  Message {num:02d} ({conf:.1f}% confidence): {msg[:60]}")
    print("=" * 65)


# Simulated organizational inbox
organizational_inbox = [
    "Free entry in 2 a wkly comp to win FA Cup final tkts!",
    "Hey can you pick up milk on your way home?",
    "WINNER! You have been selected for a cash prize of $1000!",
    "Meeting rescheduled to 3pm tomorrow, see you there.",
    "Claim your free gift now, text WIN to 80086",
    "Are you coming to the game on Saturday?",
    "You have been pre-approved for a loan of up to $50,000!",
    "Don't forget mom's birthday is next week",
    "Exclusive deal: 50% off all products this weekend only!",
    "Can you send over the project files before EOD?",
]

analyze_batch(organizational_inbox)

**Demo results discussion:**

The batch analyzer successfully identifies promotional spam messages while correctly clearing conversational and work-related messages. The structured report format, including a spam rate percentage and a dedicated flagged-messages section, makes it immediately useful for an IT security analyst who needs to present findings or take action on specific messages.

The loan pre-approval message (Message 7) is flagged at a lower confidence than obvious prize spam, reflecting the model's uncertainty about messages that could be either legitimate banking communications or spam. This nuance in confidence scoring is a feature, not a bug, it allows analysts to apply different thresholds depending on their organization's risk tolerance.

# 5. Reflection

## Evaluation Results

The fine-tuned DistilBERT model achieved 99.1% accuracy on the held-out test set of 1,115 SMS messages. This is a strong result for a binary classification task, and it reflects both the quality of the pretrained DistilBERT model and the relative clarity of the spam vs. not-spam distinction in the SMS Spam Collection dataset.

Looking at the training curve, accuracy improved from 98.9% at epoch 1 to 99.1% at epoch 2 and remained flat at epoch 3. Meanwhile, validation loss increased from epoch 2 onward (0.040 → 0.047 → 0.062), signaling the onset of overfitting. The `load_best_model_at_end=True` setting automatically retained the epoch 2 checkpoint, which represents the optimal tradeoff between fit and generalization.

For the intended use case, screening SMS messages for spam, 99.1% accuracy is production-quality performance. In a real deployment on 1,000 messages per day, roughly 9 messages would be misclassified, which is an acceptable error rate for a first-pass filter that can be supplemented with human review for borderline cases.

However, the sample prediction tests revealed an important gap: the model failed to classify a phishing-style message ("URGENT: Your bank account has been compromised. Call now!") as spam, labeling it NOT SPAM with 98% confidence. This is a significant real-world limitation, because phishing messages, not promotional spam, are the primary vector for financial fraud and data breaches in organizational settings.

## Contributions

What I built in this project goes beyond simply calling a pre-built sentiment API. I went through the full pipeline: loading and preprocessing a real labeled dataset, configuring and executing the fine-tuning process, evaluating the model against a held-out test set, deploying the model to Hugging Face Hub with a complete model card, and building two distinct applications that demonstrate the model in different real-world contexts.

The two applications, a single-message checker and a batch security report generator, represent different usage scenarios targeting different types of users. The single-message checker is designed for end users who want an accessible tool to verify suspicious messages. The batch analyzer is designed for IT security professionals who need to screen large volumes of messages and produce structured reports. Building both required thinking about how the same underlying model can be packaged differently to serve different needs.

The decision to apply this to spam detection rather than the standard movie review sentiment analysis task was deliberate. Spam detection has direct relevance to cybersecurity and IT roles, which connects to the broader Information Science curriculum. It also allowed for a more meaningful discussion of real-world limitations, specifically, the difference between promotional spam and phishing attacks.

## Limitations

**Data limitations:** The SMS Spam Collection is a well-known benchmark dataset but it has real constraints. It contains only 5,574 messages, it is heavily skewed toward promotional spam (prize offers, free gifts, competition entries) rather than phishing or social engineering attacks, and it was collected primarily from a single source (a UK university), which may not represent the diversity of spam seen in other regions or organizational contexts.

**Phishing blind spot:** The most significant operational limitation is the model's failure to detect phishing-style messages that use legitimate-sounding language. A message like "Your bank account has been compromised" doesn't use the vocabulary of promotional spam, so the model treats it as a normal message. This is a fundamental limitation of training on promotional spam data, the model learns to detect promotional patterns, not deceptive intent.

**Binary classification:** The model only distinguishes between two classes. Real-world spam comes in many categories: promotional spam, phishing, malware links, social engineering, scam calls. A more granular multi-class classifier would be more actionable for security teams.

**Language limitation:** The model is English-only. Organizations operating internationally or receiving multilingual communications would need separate models or a multilingual base model like `xlm-roberta`.

**Static training:** Once deployed, the model does not adapt to new spam patterns. Spammers constantly evolve their tactics, and a model trained in 2026 may not catch spam that emerges in 2027 without periodic retraining.

## Future Directions

The most impactful improvement would be to augment the training dataset with phishing and social engineering examples. Datasets like the Enron Email Dataset or PhishTank data could provide the phishing vocabulary the current model lacks, dramatically improving its real-world security utility.

A second direction would be moving to a multi-class classification framework with categories like promotional spam, phishing, malware, and legitimate, giving security analysts more actionable information than a binary verdict.

Explainability is another important future direction. Attention visualization tools could highlight which words most influenced each classification decision, helping analysts understand why a message was flagged and building trust in the system among non-technical users.

Finally, this pipeline could be extended to email classification, which is a larger and more consequential spam problem for organizations. The core architecture, fine-tuned DistilBERT, HuggingFace Trainer, deployed to Hub, would remain the same. The main change would be handling longer input sequences (emails are much longer than SMS messages), which might require switching to a model with a larger context window.

## Lessons Learned

This project gave me a practical, end-to-end understanding of how modern NLP systems are built and deployed. The most important lesson was that the fine-tuning process itself is accessible, the HuggingFace Trainer abstracts away the complexity of gradient descent and backpropagation, letting me focus on the data and the evaluation. The hard part was not writing the code but understanding what the results mean and where the system fails.

The phishing detection failure was the most instructive moment. It forced me to think about the difference between what a model learns (statistical patterns in training data) and what the task actually requires (detecting deceptive intent). A model can be 99% accurate on a benchmark and still be inadequate for real-world deployment if the benchmark does not capture the full range of the problem.

I also came to appreciate the importance of the confidence score alongside the binary label. A 99% confident SPAM verdict and a 55% confident SPAM verdict should trigger very different responses, and building the application to surface that distinction makes it substantially more useful than a system that just returns a label.

# 6. References

Almeida, T. A., Gomez Hidalgo, J. M., & Yamakami, A. (2011). Contributions to the study of SMS spam filtering: New collection and results. Proceedings of the 11th ACM symposium on Document engineering, 259-262. https://dl.acm.org/doi/10.1145/2034691.2034742

Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. Proceedings of NAACL-HLT 2019. https://arxiv.org/abs/1810.04805

Hugging Face. (2024). DistilBERT base model (uncased). https://huggingface.co/distilbert-base-uncased

Hugging Face. (2024). SMS Spam Collection dataset. https://huggingface.co/datasets/sms_spam

Hugging Face. (2024). Transformers documentation: Trainer. https://huggingface.co/docs/transformers/main_classes/trainer

Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: Smaller, faster, cheaper and lighter. arXiv:1910.01108. https://arxiv.org/abs/1910.01108

Shah, C. (2023). A hands-on introduction to machine learning. Cambridge University Press. https://doi.org/10.1017/9781009123303

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., & Rush, A. M. (2020). Transformers: State-of-the-art natural language processing. Proceedings of EMNLP 2020: System Demonstrations, 38-45. https://aclanthology.org/2020.emnlp-demos.6

ArefAhmad. (2026). spam-detector-distilbert [Model]. Hugging Face. https://huggingface.co/ArefAhmad/spam-detector-distilbert